# Archive Processed Source Files

This notebook moves **incremental CSV files** that have been successfully loaded into the **gold** layer to an archive folder. It checks the `control.watermark` table to confirm each entity has been successfully processed (`SUCCESS` status with a non-null `last_processed_file`) before moving files. Entities at `INITIAL_BASELINE_COMPLETE` are skipped — their incremental files have not been loaded yet. Only the `incremental/` subfolder is processed — `initial/` files are never touched. Manifest JSON files and other non-CSV files are left in the source directory.

**Archive structure:** `/Volumes/customer360_dev/staging/archive/<entity>/incremental/<filename>`

**What it does:**
1. Reads the watermark table to identify entities eligible for archival
2. Moves only `.csv` files from `source_files/<entity>/incremental/` to `archive/<entity>/incremental/`
3. Skips files already in archive (no duplicates)
4. Logs the archival to the `control.pipeline_audit` table
5. Prints a summary report

In [0]:
import os
import shutil
import uuid
from datetime import datetime
from pyspark.sql import functions as F

# ============================================================
# Configuration
# ============================================================

SOURCE_BASE = "/Volumes/customer360_dev/staging/source_files"
ARCHIVE_BASE = "/Volumes/customer360_dev/staging/archive"
WATERMARK_TABLE = "customer360_dev.control.watermark"
AUDIT_TABLE = "customer360_dev.control.pipeline_audit"
PIPELINE_NAME = "customer360_incremental_master_pipeline"
NOTEBOOK_NAME = "49_archive_processed_source_files"

# Archive date for this run (used for audit logging only)
archive_date = datetime.now().strftime("%Y%m%d")
audit_run_id = str(uuid.uuid4())
run_start_ts = datetime.now()

print(f"Archive date:  {archive_date}")
print(f"Source base:   {SOURCE_BASE}")
print(f"Archive base:  {ARCHIVE_BASE}")
print(f"Audit run ID:  {audit_run_id}")

Archive date:  20261001
Source base:   /Volumes/customer360_dev/staging/source_files
Archive base:  /Volumes/customer360_dev/staging/archive
Audit run ID:  08c49961-7f98-44d4-9f3b-253ca01f953c


In [0]:
# ============================================================
# 1. Read watermark table to determine which entities are processed
# ============================================================

watermark_df = spark.table(WATERMARK_TABLE)

# Only archive entities that have successfully processed incremental data
# INITIAL_BASELINE_COMPLETE means only the initial load was done —
# incremental files have NOT been loaded yet, so they must NOT be archived
eligible_entities = (
    watermark_df
    .filter(F.col("load_status") == "SUCCESS")
    .filter(F.col("last_processed_file").isNotNull())
    .select(
        "entity_name",
        "source_system",
        "load_status",
        "last_processed_file",
        "updated_at"
    )
    .orderBy("entity_name")
)

eligible_rows = eligible_entities.collect()
eligible_count = len(eligible_rows)

print(f"Entities eligible for archival: {eligible_count}")
print()
for row in eligible_rows:
    print(f"  {row['entity_name']:20s}  status={row['load_status']:30s}  updated={row['updated_at']}")

Entities eligible for archival: 8

  addresses             status=SUCCESS                         updated=2026-10-01 07:16:55.694238
  customers             status=SUCCESS                         updated=2026-10-01 07:17:00.541363
  order_items           status=INITIAL_BASELINE_COMPLETE       updated=2026-09-27 06:04:23.147244
  orders                status=INITIAL_BASELINE_COMPLETE       updated=2026-09-27 06:04:23.147244
  payments              status=INITIAL_BASELINE_COMPLETE       updated=2026-09-27 06:04:23.147244
  products              status=SUCCESS                         updated=2026-10-01 07:29:41.578495
  subscriptions         status=SUCCESS                         updated=2026-10-01 07:16:50.648959
  support_tickets       status=INITIAL_BASELINE_COMPLETE       updated=2026-09-27 06:04:23.147244


In [0]:
# ============================================================
# 2. Archive CSV files from INCREMENTAL folder only
#    Rules:
#      - Only .csv files from source/<entity>/incremental/ are moved
#      - Initial folder files are never touched
#      - Each file is moved individually (never whole folders)
#      - Archive mirrors source: archive/<entity>/incremental/<filename>
#      - If a file already exists in the archive it is SKIPPED (no duplication)
#      - Source folder structure is always preserved
# ============================================================

archive_results = []

for row in eligible_rows:
    entity        = row["entity_name"]
    source_system = row["source_system"]
    source_path   = os.path.join(SOURCE_BASE, entity)
    incremental_path = os.path.join(source_path, "incremental")
    archive_path  = os.path.join(ARCHIVE_BASE, entity)

    result = {
        "entity": entity,
        "source_system": source_system,
        "status": "ARCHIVED",
        "files_moved": 0,
        "files_skipped": 0,
        "total_mb": 0.0,
        "archive_path": archive_path,
        "detail": "",
    }

    if not os.path.exists(incremental_path):
        result["status"]  = "SKIPPED"
        result["detail"]  = "Incremental folder not found"
        archive_results.append(result)
        print(f"  SKIP: {entity} — incremental folder does not exist")
        continue

    files_moved   = 0
    files_skipped = 0
    total_bytes   = 0

    # Walk ONLY the incremental/ subfolder
    for root, dirs, files in os.walk(incremental_path):
        for filename in files:

            # ── Rule 1: only CSV files ──
            if not filename.endswith(".csv"):
                continue

            src_file = os.path.join(root, filename)
            rel_path = os.path.relpath(src_file, source_path)   # e.g. incremental/customers_incremental_001.csv
            dst_file = os.path.join(archive_path, rel_path)     # e.g. archive/customers/incremental/customers_incremental_001.csv

            # ── Rule 2: skip if already archived (no duplicates) ──
            if os.path.exists(dst_file):
                files_skipped += 1
                continue

            # ── Move this single file ──
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.move(src_file, dst_file)
            file_size = os.path.getsize(dst_file)
            files_moved += 1
            total_bytes += file_size
            print(f"    moved: {entity}/{rel_path}  ({file_size / (1024*1024):.1f} MB)")

    total_mb = round(total_bytes / (1024 * 1024), 1)
    result["files_moved"]   = files_moved
    result["files_skipped"] = files_skipped
    result["total_mb"]      = total_mb
    result["detail"]       = f"Moved {files_moved} CSV files ({total_mb} MB), skipped {files_skipped} duplicates"
    archive_results.append(result)

    print(f"  {entity:20s}  moved {files_moved:4d}  skipped {files_skipped:4d}  ({total_mb:10.1f} MB)")

# ── Ensure source subfolders always exist for future loads ──
for row in eligible_rows:
    entity = row["entity_name"]
    for sub in ("initial", "incremental"):
        os.makedirs(os.path.join(SOURCE_BASE, entity, sub), exist_ok=True)

# ── Totals ──
archived_count = len([r for r in archive_results if r["status"] == "ARCHIVED"])
skipped_count  = len([r for r in archive_results if r["status"] == "SKIPPED"])
total_moved    = sum(r["files_moved"]   for r in archive_results)
total_skipped  = sum(r["files_skipped"] for r in archive_results)
print(f"\nArchived: {archived_count} entities | Skipped: {skipped_count} entities")
print(f"Total CSV files moved: {total_moved} | Duplicates skipped: {total_skipped}")

  addresses             moved    0  skipped    0  (       0.0 MB)
  customers             moved    0  skipped    0  (       0.0 MB)
  order_items           moved    0  skipped    0  (       0.0 MB)
  orders                moved    0  skipped    0  (       0.0 MB)
  payments              moved    0  skipped    0  (       0.0 MB)
  products              moved    0  skipped    0  (       0.0 MB)
  subscriptions         moved    0  skipped    0  (       0.0 MB)
  support_tickets       moved    0  skipped    0  (       0.0 MB)

Archived: 8 entities | Skipped: 0 entities
Total CSV files moved: 0 | Duplicates skipped: 0


In [0]:
# ============================================================
# 3. Log archival to pipeline_audit table
# ============================================================

run_end_ts = datetime.now()

if archive_results:
    audit_records = []
    for r in archive_results:
        audit_records.append({
            "audit_run_id": audit_run_id,
            "pipeline_run_id": "",
            "entity_name": r["entity"],
            "pipeline_name": PIPELINE_NAME,
            "layer": "ARCHIVE",
            "source_system": r["source_system"],
            "source_object": os.path.join(SOURCE_BASE, r["entity"]),
            "target_object": r["archive_path"],
            "load_type": "ARCHIVE",
            "run_status": r["status"],
            "start_timestamp": run_start_ts,
            "end_timestamp": run_end_ts,
            "input_row_count": 0,
            "output_row_count": r["files_moved"],
            "rejected_row_count": r.get("files_skipped", 0),
            "insert_count": 0,
            "update_count": 0,
            "unchanged_count": 0,
            "reconciliation_status": "",
            "error_class": "",
            "error_message": r["detail"] if r["status"] == "SKIPPED" else "",
            "notebook_name": NOTEBOOK_NAME,
            "created_at": run_end_ts,
            "updated_at": run_end_ts,
        })

    audit_df = spark.createDataFrame(audit_records)

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

    print(f"Logged {len(audit_records)} archival records to {AUDIT_TABLE}")
else:
    print("No files archived - skipping audit log")

Logged 8 archival records to customer360_dev.control.pipeline_audit


In [0]:
# ============================================================
# 4. Summary report
# ============================================================

print("=" * 70)
print("ARCHIVE SUMMARY")
print("=" * 70)
print(f"Run ID:         {audit_run_id}")
print(f"Archive date:   {archive_date}")
print(f"Started:        {run_start_ts}")
print(f"Completed:      {run_end_ts}")
print(f"Duration:       {(run_end_ts - run_start_ts).total_seconds():.1f} seconds")
print()
print(f"{'Entity':<20s} {'Status':<10s} {'Moved':>6s} {'Skipped':>8s} {'Size (MB)':>10s}")
print("-" * 60)
for r in archive_results:
    print(f"{r['entity']:<20s} {r['status']:<10s} {r['files_moved']:>6d} {r.get('files_skipped', 0):>8d} {r['total_mb']:>10.1f}")
print("-" * 60)
total_files = sum(r["files_moved"] for r in archive_results)
total_skipped = sum(r.get("files_skipped", 0) for r in archive_results)
total_mb = sum(r["total_mb"] for r in archive_results)
print(f"{'TOTAL':<20s} {'':<10s} {total_files:>6d} {total_skipped:>8d} {total_mb:>10.1f}")
print("=" * 70)
print(f"\nArchive location: {ARCHIVE_BASE}/<entity>/initial/ and {ARCHIVE_BASE}/<entity>/incremental/")

ARCHIVE SUMMARY
Run ID:         08c49961-7f98-44d4-9f3b-253ca01f953c
Archive date:   20261001
Started:        2026-10-01 08:43:21.402100
Completed:      2026-10-01 08:43:40.516890
Duration:       19.1 seconds

Entity               Status      Moved  Skipped  Size (MB)
------------------------------------------------------------
addresses            ARCHIVED        0        0        0.0
customers            ARCHIVED        0        0        0.0
order_items          ARCHIVED        0        0        0.0
orders               ARCHIVED        0        0        0.0
payments             ARCHIVED        0        0        0.0
products             ARCHIVED        0        0        0.0
subscriptions        ARCHIVED        0        0        0.0
support_tickets      ARCHIVED        0        0        0.0
------------------------------------------------------------
TOTAL                                0        0        0.0

Archive location: /Volumes/customer360_dev/staging/archive/<entity>/initial/ 